# Baseline and Random Forest Models

This notebook establishes predictive benchmarks for the Cook County housing-price task. The primary evaluation set contains **2019 pure-market sales for properties not observed during 2013–2018**, so performance reflects forward generalization to unseen properties rather than repeated-parcel memorization.

The modeling sequence is deliberately incremental: naive baseline → structural Random Forest → price-level calibration → location-aware Random Forest. Census demographic attributes remain excluded from prediction throughout.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import build_analysis_dataset
from src.features import PRIMARY_FEATURES, define_modeling_population, create_temporal_splits, make_model_matrices
from src.modeling import (
    LOCATION_FEATURES, fit_dummy_baseline, predict_dummy_dollars,
    build_random_forest_pipeline, fit_log_target_model, predict_dollars,
)
from src.evaluation import regression_metrics, metrics_table, price_decile_calibration

DATA_ZIP = PROJECT_ROOT / 'data' / 'raw' / 'cook_county_data.zip'

## 1. Reconstruct the Modeling Sample

The same pure-market population and temporal split defined in Notebook 2 are reconstructed here so every model is evaluated on a fixed holdout population.

In [ ]:
data_fair = build_analysis_dataset(DATA_ZIP)
model_data = define_modeling_population(data_fair)
train_data, test_data, repeated_property_test = create_temporal_splits(model_data)
matrices = make_model_matrices(train_data, test_data, repeated_property_test)

X_train, X_test = matrices['X_train'], matrices['X_test']
X_test_repeated = matrices['X_test_repeated']
y_train, y_test = matrices['y_train'], matrices['y_test']
y_test_repeated, y_train_log = matrices['y_test_repeated'], matrices['y_train_log']

print(f'Training sales: {len(X_train):,}')
print(f'Primary unseen-property test sales: {len(X_test):,}')
print(f'Repeated-property test sales: {len(X_test_repeated):,}')
print(f'Structural predictors: {len(PRIMARY_FEATURES)}')

## 2. Naive Regression Baseline

A median dummy model establishes the minimum performance bar. It is fit on the log-price target and transformed back to dollars for evaluation.

In [ ]:
dummy_model = fit_dummy_baseline(y_train_log)
dummy_pred = predict_dummy_dollars(dummy_model, len(X_test))
pd.Series(regression_metrics(y_test, dummy_pred), name='Naive baseline')

## 3. Structural Random Forest

The primary Random Forest uses structural and transaction-timing features while excluding fine-grained geography and all ACS demographic variables. Preprocessing is fit inside the training pipeline, and the target remains `log(Sale Price)`.

In [ ]:
structural_rf = build_random_forest_pipeline(include_location=False)
structural_rf = fit_log_target_model(structural_rf, X_train, y_train_log)

structural_pred_train = predict_dollars(structural_rf, X_train)
structural_pred = predict_dollars(structural_rf, X_test)
structural_pred_repeated = predict_dollars(structural_rf, X_test_repeated)

structural_results = pd.DataFrame({
    'Training': regression_metrics(y_train, structural_pred_train),
    'Primary unseen-property test': regression_metrics(y_test, structural_pred),
    'Repeated-property test': regression_metrics(y_test_repeated, structural_pred_repeated),
}).T
structural_results.style.format({'MAE':'${:,.0f}','RMSE':'${:,.0f}','Median AE':'${:,.0f}','MAPE':'{:.1f}%','R2':'{:.3f}'})

In [ ]:
comparison = metrics_table({
    'Naive baseline': (y_test, dummy_pred),
    'Structural Random Forest': (y_test, structural_pred),
})
comparison.style.format({'MAE':'${:,.0f}','RMSE':'${:,.0f}','Median AE':'${:,.0f}','MAPE':'{:.1f}%','R2':'{:.3f}'})

### Structural-model result

The completed full-project run produced approximately **$181.7K MAE, $338.6K RMSE, and 91.9% MAPE** for the naive baseline. The structural Random Forest improved to approximately **$109.2K MAE, $181.2K RMSE, $73.2K median absolute error, 60.5% MAPE, and R² = 0.675** on the unseen-property 2019 holdout.

## 4. Price-Level Calibration

Aggregate accuracy can conceal systematic price-dependent errors. We therefore group the primary test set by actual sale-price decile and inspect median signed percentage error within each group.

In [ ]:
structural_calibration = price_decile_calibration(y_test, structural_pred)
structural_calibration

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, structural_pred, alpha=0.25, s=12)
plot_min = min(y_test.min(), structural_pred.min())
plot_max = max(y_test.max(), structural_pred.max())
plt.plot([plot_min, plot_max], [plot_min, plot_max], linestyle='--')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Actual Sale Price ($)'); plt.ylabel('Predicted Sale Price ($)')
plt.title('Structural Random Forest: Actual vs. Predicted')
plt.tight_layout(); plt.show()

The structural Random Forest exhibits strong regression toward the center of the market. In the prior full run, the lowest sale-price decile was overpredicted by a median **216.5%**, while the highest decile was underpredicted by approximately **27.4%**. This matters for the later fairness audit because subgroup disparities can partly reflect differences in underlying property-value distributions.

## 5. Location-Aware Random Forest Benchmark

Location is a major determinant of market value, but it may also encode socioeconomic and residential-segregation patterns. A secondary benchmark therefore adds **only latitude and longitude** to the structural feature set while continuing to exclude Census demographic variables.

In [ ]:
location_features = PRIMARY_FEATURES + LOCATION_FEATURES
X_train_location = train_data[location_features].copy()
X_test_location = test_data[location_features].copy()
X_test_repeated_location = repeated_property_test[location_features].copy()

location_rf = build_random_forest_pipeline(include_location=True)
location_rf = fit_log_target_model(location_rf, X_train_location, y_train_log)
location_pred = predict_dollars(location_rf, X_test_location)
location_pred_repeated = predict_dollars(location_rf, X_test_repeated_location)

In [ ]:
rf_comparison = metrics_table({
    'Structural Random Forest': (y_test, structural_pred),
    'Location-aware Random Forest': (y_test, location_pred),
})
rf_comparison.style.format({'MAE':'${:,.0f}','RMSE':'${:,.0f}','Median AE':'${:,.0f}','MAPE':'{:.1f}%','R2':'{:.3f}'})

### Predictive value of geography

In the completed full-project run, adding latitude and longitude reduced MAE from approximately **$109.2K to $75.1K**, RMSE from **$181.2K to $141.2K**, median absolute error from **$73.2K to $45.2K**, and MAPE from **60.5% to 33.9%**. R² increased from **0.675 to 0.802**.

The magnitude of this improvement demonstrates that geography contains substantial market information beyond observed structural characteristics. It also motivates the project's central fairness tradeoff: an accuracy-improving feature can still encode spatial patterns associated with socioeconomic inequality.

## 6. Structural vs. Location-Aware Calibration

The final comparison asks whether geography improves not only aggregate accuracy but also calibration across the sale-price distribution.

In [ ]:
location_calibration = price_decile_calibration(y_test, location_pred)
calibration_comparison = pd.DataFrame({
    'price_decile': range(1, len(structural_calibration) + 1),
    'median_actual_price': structural_calibration['median_actual'].values,
    'Structural RF': structural_calibration['median_signed_pct_error'].values,
    'Location-aware RF': location_calibration['median_signed_pct_error'].values,
})
calibration_comparison

In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(calibration_comparison['price_decile'], calibration_comparison['Structural RF'], marker='o', label='Structural RF')
plt.plot(calibration_comparison['price_decile'], calibration_comparison['Location-aware RF'], marker='o', label='Location-aware RF')
plt.axhline(0, linestyle='--', linewidth=1)
plt.xlabel('Actual Sale Price Decile'); plt.ylabel('Median Percentage Error (%)')
plt.title('Prediction Calibration Across the Price Distribution')
plt.xticks(range(1, 11)); plt.legend(); plt.tight_layout(); plt.show()

## 7. Takeaways

The structural Random Forest substantially outperforms the naive baseline, and adding only latitude/longitude yields another large accuracy gain. However, both models retain price-dependent calibration errors. The next notebook asks whether a different tree-learning algorithm—Histogram Gradient Boosting—can improve both aggregate accuracy and calibration while using the same location-aware information set.